In [28]:
import pandas as pd 
import plotly.express as px
import random 


#definindo bicho
bicho = {
    "data_nascimento":0,
    "vivo":True,
    "expectativa_de_vida":3,
    "anos_de_vida":None,
    "data_morte":None,
    "dieta":None 
    
}


#definindo floresta
quantidade_bichos_inicial = 10
grama = 100 
crescimento_grama = 10
lista_grama = [{'ano':0,'quantidade_grama':grama}]
lista_bichos = []
dieta_lista = ["herbivoro","carnivoro"]

for _ in range(quantidade_bichos_inicial):
    bicho_inicial = bicho.copy()
    dieta = random.choices(dieta_lista,weights=[2,1])[0]
    anos_de_vida = round(random.gauss(mu=bicho["expectativa_de_vida"], sigma=1.0))
    bicho_inicial['anos_de_vida'] = anos_de_vida
    bicho_inicial["dieta"] = dieta
    lista_bichos.append(bicho_inicial)

df_estado_inicial = pd.DataFrame(lista_bichos)
df_inicial_dieta = df_estado_inicial.groupby("dieta")["vivo"].count()

print("distribuicao inicial dieta:")
print(df_inicial_dieta)

#loop dos anos
max_anos = 1000

for ano_atual in range(1,max_anos+1):
    novos_bichos = []
    lista_bichos_vivos = [b for b in lista_bichos if b["vivo"]]
    for bicho in lista_bichos_vivos:

        idade = ano_atual - bicho["data_nascimento"]

        anos_de_vida = bicho['anos_de_vida']
        if idade >= anos_de_vida:
            bicho['vivo'] = False
            bicho['data_morte'] = ano_atual

        else:
            dieta = bicho["dieta"]
            if dieta == "carnivoro":
                presa = random.choice(lista_bichos_vivos)
                
                dieta_presa = presa["dieta"]

                if dieta_presa == "carnivoro":
                    continue
                else:
                    presa["vivo"] = False
                    presa["data_morte"] = ano_atual
                    novo_bicho = bicho.copy()
                    novo_bicho["data_nascimento"] = ano_atual
                    novos_bichos.append(novo_bicho)
            else:  # é herbivoro   
                chance_reproducao = random.randint(0,100)
                if chance_reproducao <= grama:
                    novo_bicho = bicho.copy()
                    novo_bicho["data_nascimento"] = ano_atual
                    novos_bichos.append(novo_bicho)
                    grama -= 1                   
                else:
                    continue

    info_grama = {'ano':ano_atual,'quantidade_grama':grama}
    lista_grama.append(info_grama)
    grama += crescimento_grama

    lista_bichos.extend(novos_bichos)

df_bichos = pd.DataFrame(lista_bichos)

df_grama = pd.DataFrame(lista_grama)

#adicionando coluna de idade 

df_bichos["idade"] = df_bichos["data_morte"] - df_bichos["data_nascimento"]
df_bichos.loc[df_bichos["idade"].isna(),"idade"] = max_anos - df_bichos.loc[df_bichos["idade"].isna(),"data_nascimento"] 

#analises 
def contagem_total_grafico(df):

    # conta quantos bichos nasceram em cada data
    contagem = (
        df
        .groupby("data_nascimento")
        .size()
        .reset_index(name="qtd_bichos")          # renomeia a coluna de contagem
        .sort_values("data_nascimento")         # garante ordenação
    )

    # soma acumulada
    contagem["total_acumulado"] = contagem["qtd_bichos"].cumsum()

    fig = px.line(
        contagem,
        x="data_nascimento",
        y="total_acumulado",
        title="contagem_de_bixos"
    )

    fig.show()



df_bichos["idade"].describe()
df_grama['quantidade_grama'].describe()

distribuicao inicial dieta:
dieta
carnivoro    1
herbivoro    9
Name: vivo, dtype: int64


count    1001.000000
mean       37.808192
std        14.178687
min         6.000000
25%        27.000000
50%        36.000000
75%        46.000000
max       100.000000
Name: quantidade_grama, dtype: float64

In [ ]:
def populacao_ano(df_bichos,df_grama):


    nascimento_ano = df_bichos.groupby(["data_nascimento","dieta"]).agg(nascimentos=("dieta","count")).reset_index()
    morte_ano = df_bichos.groupby(["data_morte","dieta"]).agg(mortes=("dieta","count")).reset_index()
    df_censo = pd.merge(nascimento_ano,morte_ano,how="outer",left_on=["data_nascimento","dieta"],right_on=["data_morte","dieta"])

    df_censo = df_censo.rename(columns={"data_nascimento":"ano"})
    df_censo.loc[df_censo["ano"].isna(),"ano"] = df_censo.loc[df_censo["ano"].isna(),"data_morte"] 

    df_censo.drop(columns=["data_morte"],inplace=True)

    start_year = int(df_censo['ano'].min())
    end_year = int(df_censo['ano'].max())

    all_years = range(start_year, end_year + 1)
    all_categories = df_censo['dieta'].unique()

    new_index = pd.MultiIndex.from_product(
        [all_categories, all_years], 
        names=['dieta', 'ano']
    )

    df_censo = (
        df_censo.set_index(['dieta', 'ano'])
        .reindex(new_index)
        .reset_index()
    )
    df_censo = df_censo.fillna(0,inplace=False)

    ordem_das_colunas = ["ano","dieta","nascimentos","mortes"]
    df_censo = df_censo[ordem_das_colunas]
    df_censo = df_censo.sort_values(by="ano")
    df_censo["natalidade"] = df_censo["nascimentos"] - df_censo["mortes"]

    df_censo["populacao"] = df_censo.groupby(["dieta"])["natalidade"].transform("cumsum")

    fig = px.line(
        df_censo,
        x="ano",
        y="populacao",
        color= "dieta",
       title="populacao"
    )

    fig.add_scatter(
        x=df_grama["ano"],
        y=df_grama["quantidade_grama"],
        mode="lines+markers",
        name="Quantidade de grama"
    )

    fig.show()

    return df_censo


    


populacao_ano(df_bichos,df_grama)


In [ ]:

df.loc[df["idade"]<0]